### Qu: Top 3 salaries of that department. 

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW employees AS 

SELECT *  

FROM VALUES  

(1, 'Alice', 'Engineering', 95000), 

(2, 'Bob', 'Engineering', 88000), 

(3, 'Charlie', 'Engineering', 88000), 

(4, 'David', 'Engineering', 102000), 

(7, 'Grace', 'Sales', 72000), 

(8, 'Heidi', 'Sales', 91000), 

(9, 'Ivan', 'Sales', 58000), 

(12, 'Liam', 'HR', 55000), 

(13, 'Mia', 'HR', 61000), 

(16, 'Paul', 'HR', 65000), 

(17, 'Quinn', 'Marketing', 80000), 

(18, 'Ruth', 'Marketing', 80000), 

(19, 'Sam', 'Marketing', 95000), 

(20, 'Tina', 'Marketing', 73000), 

(20, 'Test', 'Marketing', NULL) 

AS employees(emp_id, emp_name, department, salary); 

In [0]:
%sql
select * from employees

In [0]:
%sql
select emp_name, department,salary, rank() over (partition by department order by salary desc NULLs last) as rn from employees
qualify (rn<4)

In [0]:
%sql
select emp_name, department,salary from employees e1
where (select count(*) from employees e2 where e1.department=e2.department and e1.salary<e2.salary) <3
and salary is not null
order by department,salary desc

In [0]:
%sql
/* Top 3 per department via self-join */
SELECT
  e1.department,
  e1.emp_id,
  e1.emp_name,
  e1.salary
FROM employees AS e1
LEFT JOIN employees AS e2
  ON e2.department = e1.department
  AND e2.salary > e1.salary
GROUP BY all
HAVING COUNT(e2.emp_id) < 3

### For each token owner, pull only the single most recently used record.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW token_activity AS

SELECT *
FROM VALUES
(1043,197,'write','revoked','2025-02-02','2027-02-02','2026-02-02',47),
(1086,294,'admin','expired','2025-03-03','2027-03-03','2026-03-03',94),
(1129,391,'read:users','Active','2025-04-04','2027-04-04','2026-04-04',141),
(1172,488,'write:orders','REVOKED','2025-05-05','2027-05-05','2026-05-05',188),
(1215,585,'read:analytics','active','2025-06-06','2027-06-06','2026-06-06',235),
(1258,682,'full_access','revoked','2025-07-07',NULL,'2026-07-07',282),
(1301,779,'read','expired','2025-08-08','2027-08-08','2026-08-08',329),
(1344,876,'write','Active','2025-09-09','2027-09-09',NULL,0),
(1387,973,'admin','REVOKED','2025-10-10','2027-10-10','2026-10-10',423),
(1430,1070,'read:users','active','2025-11-11','2027-11-11','2026-11-11',470)
AS token_activity(
    token_id,
    owner_id,
    scope,
    status,
    issued,
    expires,
    last_used,
    requests
);

In [0]:
%sql
select * from token_activity

In [0]:
%sql
SELECT
  department_id,
  employee_id,
  salary
FROM employees AS e1
WHERE (
  SELECT
    COUNT(*)
  FROM employees AS e2
  WHERE e2.department_id = e1.department_id
  AND (
  e2.salary > e1.salary
  OR (
  e2.salary = e1.salary
  AND e2.employee_id < e1.employee_id
)
)
) < 3